# Занятие 9. Лабораторная: стекинг, AdaBoost и градиентный бустинг

Срок сдачи — 7 ноября 2026 года, ноутбук присылают преподавателю.
В лабораторной пятнадцать задач: двенадцать обязательных и три бонусные,
у бонусных это написано в заголовке. Обязательные задачи дают до 10 баллов;
задачи 8 и 9 весят вдвое больше остальных. Каждая бонусная задача дает
до балла сверху пропорционально доле пройденных проверок; итог округляется
вниз и не больше 11.

Цель — написать ансамбли трех видов. Сначала стекинг и блендинг: метамодель
на предсказаниях базовых моделей вне блоков и на отложенной части выборки. Затем
AdaBoost с весами объектов. Затем градиентный бустинг целиком: функции потерь и их
антиградиенты, начальное приближение, значения листьев, регрессор и бинарный
классификатор с интерфейсом `fit` и `predict`; темп обучения, ранняя остановка
и сравнение своей реализации с библиотеками в одинаковых условиях. В бонусах —
многоклассовый бустинг, разбиение по второй производной, как в XGBoost,
и квантильная регрессия.

**Заготовки.** В каждой заготовке указано, что должно получиться: тип
и форма в аннотации, размеры входов и выходов в комментарии перед функцией.
Как считать — в теории перед задачей, в семинаре и в конспекте занятия.
Имена переменных, функций, классов и атрибутов менять нельзя, проверка ищет
их по именам. Вспомогательные функции можно добавлять свои. После каждой
задачи идет ячейка с открытыми проверками; при сдаче работа дополнительно
проверяется закрытыми тестами на других данных.

**Что можно использовать.** Базовые модели — из sklearn: в стекинге и блендинге
`LinearRegression`, `KNeighborsRegressor` и `DecisionTreeRegressor`, в AdaBoost — дерево
классификации `DecisionTreeClassifier`, в градиентном бустинге — дерево регрессии
`DecisionTreeRegressor`: свое дерево написано в лабораторной 8. Разбиения выборки —
`KFold` и `train_test_split`, копии моделей — `clone`. В задачах 1–11 и в бонусах нельзя
вызывать ансамбли sklearn (`Stacking*`, `Voting*`, `AdaBoost*`, `GradientBoosting*`,
`HistGradientBoosting*`, `RandomForest*`), `cross_val_predict`, LightGBM, CatBoost
и XGBoost; numpy и scipy использовать можно. Библиотеки нужны только в проверках
и для сравнения в задаче 12.

**Данные.** Диабет (`load_diabetes`) для регрессии, как в лабораторной 8,
опухоли груди (`load_breast_cancer`, класс 1 — доброкачественная) для бинарной
классификации, вина (`load_wine`) для многоклассовой. В каждом наборе
30% объектов — тестовый сет, а оставшиеся 70% делятся на часть для обучения
(`fit`) и валидационную (`val`) в отношении 3 : 1.

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from scipy.special import expit
from sklearn.base import clone
from sklearn.datasets import load_breast_cancer, load_diabetes, load_wine
from sklearn.ensemble import (
    AdaBoostClassifier, GradientBoostingClassifier, GradientBoostingRegressor, HistGradientBoostingClassifier,
    RandomForestClassifier, StackingRegressor,
)
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
from sklearn.model_selection import KFold, cross_val_predict, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.axisbelow": True,
})

BLUE, BLACK, OCHRE, GREY = "#0072CE", "#0F1418", "#C98A3C", "#9CA3AF"
SEED = 42


def split_three(X, y, stratify):
    """Тестовый сет 30%, остаток делится на fit и val в отношении 3 : 1."""
    strata = y if stratify else None
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=SEED, stratify=strata)

    strata_train = y_train if stratify else None
    X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=SEED,
                                                  stratify=strata_train)
    parts = (X_train, X_test, y_train, y_test, X_fit, X_val, y_fit, y_val)
    return parts


diabetes = load_diabetes()
Xd_train, Xd_test, yd_train, yd_test, Xd_fit, Xd_val, yd_fit, yd_val = split_three(
    diabetes.data, diabetes.target, stratify=False
)

cancer = load_breast_cancer()
Xc_train, Xc_test, yc_train, yc_test, Xc_fit, Xc_val, yc_fit, yc_val = split_three(
    cancer.data, cancer.target, stratify=True
)

wine = load_wine()
Xw_train, Xw_test, yw_train, yw_test, _, _, _, _ = split_three(wine.data, wine.target, stratify=True)

print("диабет:", Xd_fit.shape, Xd_val.shape, Xd_test.shape)
print("опухоли:", Xc_fit.shape, Xc_val.shape, Xc_test.shape)
print("вина:", Xw_train.shape, Xw_test.shape)

## Блок A. Стекинг и блендинг

Базовые модели $a_1, \dots, a_M$ разных типов объединяет метамодель $g$:
$a(x) = g\big(a_1(x), \dots, a_M(x)\big)$. Метамодель обучают на **метапризнаках** — предсказаниях
базовых моделей на объектах, которых эти модели не видели при обучении (семинар, раздел 2).
Данные блока — диабет: обучение на `Xd_train`, тестовый сет `Xd_test`. Базовые модели задач 2 и 3:

In [ ]:
def make_base_models() -> list:
    """Базовые модели задач 2 и 3: линейная регрессия, kNN и неглубокое дерево."""
    models = [
        LinearRegression(),
        KNeighborsRegressor(n_neighbors=15),
        DecisionTreeRegressor(max_depth=3, random_state=SEED),
    ]
    return models

### Задача 1. Предсказания вне блоков

Функция `out_of_fold_predictions` делит выборку разбиением
`KFold(n_splits=n_splits, shuffle=True, random_state=random_state)`. Для каждого блока она обучает
копию модели (`clone(model)`) на остальных блоках и записывает ее предсказания для объектов блока.
Результат — вектор предсказаний в исходном порядке объектов: у каждого объекта предсказание
модели, обученной без его блока. Модель на входе остается необученной.

In [ ]:
# model: необученная модель регрессии sklearn; X: (n, p), y: (n,)
# возвращает (n,): для каждого объекта предсказание копии модели, обученной без его блока
def out_of_fold_predictions(model, X: np.ndarray, y: np.ndarray, n_splits: int, random_state: int) -> np.ndarray:
    return ...

In [ ]:
# --- проверка ---
for model in make_base_models():
    mine = out_of_fold_predictions(model, Xd_train, yd_train, n_splits=5, random_state=SEED)
    reference = cross_val_predict(clone(model), Xd_train, yd_train, cv=KFold(5, shuffle=True, random_state=SEED))
    assert mine.shape == (len(yd_train),), "по одному предсказанию на объект"
    assert np.allclose(mine, reference), f"{type(model).__name__}: предсказания не совпали с cross_val_predict"

untouched = LinearRegression()
out_of_fold_predictions(untouched, Xd_train, yd_train, n_splits=5, random_state=SEED)
assert not hasattr(untouched, "coef_"), "модель на входе обучать не нужно: обучают ее копии"
print("проверки пройдены")

### Задача 2. Стекинг

Класс `MyStackingRegressor`. Метод `fit`:

1. для каждой базовой модели из `base_models` считает столбец предсказаний вне блоков (задача 1)
   с `self.n_splits` и `self.random_state`; столбцы в порядке `base_models` образуют матрицу
   метапризнаков `self.meta_features_`;
2. обучает на метапризнаках копию `meta_model` — `self.meta_model_`;
3. обучает на всей выборке копии базовых моделей — список `self.base_models_`.

Метод `predict` подает метамодели предсказания `base_models_` на новых объектах.

In [ ]:
class MyStackingRegressor:
    def __init__(self, base_models: list, meta_model, n_splits: int = 5, random_state: int | None = None):
        self.base_models = base_models
        self.meta_model = meta_model
        self.n_splits = n_splits
        self.random_state = random_state

    # X: (n, p), y: (n,); задает атрибуты и возвращает self:
    #   meta_features_: np.ndarray (n, M) — предсказания вне блоков, столбец на базовую модель
    #   meta_model_ — обученная копия meta_model
    #   base_models_: list длины M — копии базовых моделей, обученные на всей выборке
    def fit(self, X: np.ndarray, y: np.ndarray) -> "MyStackingRegressor":
        ...
        return self

    # X: (m, p); возвращает (m,)
    def predict(self, X: np.ndarray) -> np.ndarray:
        return ...

In [ ]:
# --- проверка ---
my_stacking = MyStackingRegressor(make_base_models(), LinearRegression(), n_splits=5, random_state=SEED)
my_stacking.fit(Xd_train, yd_train)

reference_estimators = [(f"base_{number}", model) for number, model in enumerate(make_base_models())]
reference_stacking = StackingRegressor(reference_estimators, final_estimator=LinearRegression(),
                                       cv=KFold(5, shuffle=True, random_state=SEED))
reference_stacking.fit(Xd_train, yd_train)

assert my_stacking.meta_features_.shape == (len(yd_train), 3), "метапризнаки — матрица (n, M)"
assert len(my_stacking.base_models_) == 3, "по обученной копии на базовую модель"
assert np.allclose(my_stacking.predict(Xd_test), reference_stacking.predict(Xd_test)), \
    "предсказания не совпали со StackingRegressor"
print("веса метамодели:", my_stacking.meta_model_.coef_.round(2), "свободный член:", round(my_stacking.meta_model_.intercept_, 1))
print("проверки пройдены")

### Задача 3. Блендинг и сравнение

Класс `MyBlendingRegressor`. Метод `fit` делит выборку один раз,
`train_test_split(X, y, test_size=self.blend_size, random_state=self.random_state)`: копии базовых
моделей обучает на первой части, копию метамодели — на их предсказаниях на второй части.
Атрибуты `base_models_` и `meta_model_` и метод `predict` — как в задаче 2.

Затем заполните таблицу `ensemble_table`: MSE на тестовом сете `Xd_test` для строк `ENSEMBLE_ROWS`.
Все модели обучают на `Xd_train`, метамодель — `LinearRegression()`:

- три базовые модели из `make_base_models()` по отдельности;
- среднее их предсказаний;
- метамодель, обученная на предсказаниях базовых моделей на той же выборке, на которой
  обучены базовые модели;
- стекинг: `MyStackingRegressor` с `n_splits=5` и `random_state=SEED`;
- блендинг: `MyBlendingRegressor` с `blend_size=0.3` и `random_state=SEED`.

In [ ]:
ENSEMBLE_ROWS = [
    "линейная регрессия", "kNN, k = 15", "дерево глубины 3", "среднее трех моделей",
    "метамодель на предсказаниях обучения", "стекинг", "блендинг",
]

In [ ]:
class MyBlendingRegressor:
    def __init__(self, base_models: list, meta_model, blend_size: float = 0.3, random_state: int | None = None):
        self.base_models = base_models
        self.meta_model = meta_model
        self.blend_size = blend_size
        self.random_state = random_state

    # X: (n, p), y: (n,); задает base_models_ и meta_model_, возвращает self
    def fit(self, X: np.ndarray, y: np.ndarray) -> "MyBlendingRegressor":
        ...
        return self

    # X: (m, p); возвращает (m,)
    def predict(self, X: np.ndarray) -> np.ndarray:
        return ...


ensemble_table: pd.DataFrame = ...     # (7, 1): индекс ENSEMBLE_ROWS, столбец "MSE на тесте"

In [ ]:
# --- проверка ---
assert list(ensemble_table.index) == ENSEMBLE_ROWS and list(ensemble_table.columns) == ["MSE на тесте"], \
    "индекс — ENSEMBLE_ROWS, столбец «MSE на тесте»"

linear_reference = LinearRegression().fit(Xd_train, yd_train).predict(Xd_test)
linear_mse = np.mean((linear_reference - yd_test) ** 2)
assert np.isclose(ensemble_table.loc["линейная регрессия", "MSE на тесте"], linear_mse), \
    "MSE линейной регрессии не совпала с пересчетом"

my_blending = MyBlendingRegressor(make_base_models(), LinearRegression(), blend_size=0.3, random_state=SEED)
my_blending.fit(Xd_train, yd_train)
assert len(my_blending.base_models_) == 3, "по обученной копии на базовую модель"
assert np.isclose(ensemble_table.loc["блендинг", "MSE на тесте"], np.mean((my_blending.predict(Xd_test) - yd_test) ** 2)), \
    "MSE блендинга не совпала с пересчетом"

ensemble_table.style.format(precision=0).background_gradient(cmap="Greens_r")

**Вопрос.** Почему метамодель, обученная на предсказаниях базовых моделей на обучающей
выборке, ошибается сильнее стекинга? Сравните ее веса `coef_` с весами метамодели стекинга
из задачи 2: какой базовой модели она дает больший вес и почему? Почему блендинг на этих
данных хуже стекинга?

*Ответ пишите здесь.*

## Блок B. AdaBoost

AdaBoost (adaptive boosting) — бустинг классификаторов $b_t(x) \in \{-1, +1\}$ с экспоненциальной
функцией потерь $L(y, a) = e^{-y\,a}$ и метками $y \in \{-1, +1\}$; вывод — в конспекте, раздел 6.
Алгоритм:

1. веса объектов $w_i = \frac{1}{n}$;
2. для $t = 1, \dots, T$:
   - пень `DecisionTreeClassifier(max_depth=1, random_state=self.random_state)` обучают на метках $\pm 1$
     с весами объектов (`sample_weight`);
   - взвешенная доля ошибок $\text{err}_t = \sum_{i:\, b_t(x_i) \ne y_i} w_i \,\big/\, \sum_i w_i$ и вес пня
     $\alpha_t = \frac{1}{2}\ln\frac{1 - \text{err}_t}{\text{err}_t}$;
   - веса пересчитывают, $w_i \leftarrow w_i\, e^{-\alpha_t y_i b_t(x_i)}$, и нормируют на сумму 1;
3. оценка $a(x) = \sum_t \alpha_t b_t(x)$; класс 1 при $a(x) > 0$, иначе класс 0.

### Задача 4. AdaBoost

Класс `MyAdaBoostClassifier` получает метки 0 и 1 и сам переводит их в $\pm 1$. На данных
лабораторной $0 < \text{err}_t < 0.5$ на каждом шаге.

In [ ]:
class MyAdaBoostClassifier:
    def __init__(self, n_estimators: int = 50, random_state: int | None = None):
        self.n_estimators = n_estimators
        self.random_state = random_state

    # X: (n, p), y: (n,) метки 0 и 1; задает атрибуты и возвращает self:
    #   estimators_: list[DecisionTreeClassifier] длины n_estimators — пни в порядке обучения
    #   weights_: list длины n_estimators — веса объектов (n,), с которыми обучен каждый пень
    #   alphas_: np.ndarray (n_estimators,) — веса пней α_t
    def fit(self, X: np.ndarray, y: np.ndarray) -> "MyAdaBoostClassifier":
        ...
        return self

    # X: (m, p); возвращает (m,): оценки a(x) = Σ α_t b_t(x)
    def decision_function(self, X: np.ndarray) -> np.ndarray:
        return ...

    # X: (m, p); возвращает (m,): метки 0 и 1
    def predict(self, X: np.ndarray) -> np.ndarray:
        return ...

In [ ]:
# --- проверка ---
my_ada = MyAdaBoostClassifier(n_estimators=50, random_state=SEED)
my_ada.fit(Xc_fit, yc_fit)

reference_ada = AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=50, learning_rate=1.0,
                                   random_state=SEED)
reference_ada.fit(Xc_fit, yc_fit)

assert len(my_ada.estimators_) == 50 and my_ada.alphas_.shape == (50,), "по пню и весу на шаг"
assert np.allclose(my_ada.weights_[0], 1 / len(yc_fit)), "на первом шаге веса равны 1/n"
assert np.allclose([weights.sum() for weights in my_ada.weights_], 1), "веса нормированы на сумму 1"
assert np.allclose(reference_ada.estimator_weights_, 2 * my_ada.alphas_), \
    "в sklearn вес пня вдвое больше α_t: ln((1 − err) / err)"
assert np.array_equal(my_ada.predict(Xc_fit), reference_ada.predict(Xc_fit)), \
    "предсказания на обучающей выборке не совпали с AdaBoostClassifier"

validation_accuracy = accuracy_score(yc_val, my_ada.predict(Xc_val))
print(f"accuracy на валидации: {validation_accuracy:.3f}")
print("проверки пройдены")

**Вопрос.** Как меняются веса объектов от шага к шагу? Найдите объекты, у которых
в `weights_[-1]` наибольшие веса, и объясните, почему именно у них. Сравните сумму
$\alpha_t$ первых десяти и последних десяти пней: что это говорит о вкладе поздних пней?

*Ответ пишите здесь.*

## Блок C. Функции потерь

Предсказание модели на объекте — вещественное число $a$. Три функции потерь:

- квадратичная, `"squared_error"`: $L(y, a) = \frac{1}{2}(y - a)^2$;
- абсолютная, `"absolute_error"`: $L(y, a) = |y - a|$;
- логистическая, `"log_loss"`: $y \in \{0, 1\}$, $a$ — оценка класса 1 (логарифм отношения
  шансов), $p = \sigma(a) = \frac{1}{1 + e^{-a}}$ (в scipy — `expit`),
  $L(y, a) = -y\ln p - (1 - y)\ln(1 - p) = \ln(1 + e^{a}) - y\,a$.

Последнее равенство получают подстановкой $\ln p = -\ln(1 + e^{-a})$
и $\ln(1 - p) = -a - \ln(1 + e^{-a})$.

### Задача 5. Потери и антиградиенты

Функция `loss_value` возвращает среднюю потерю по объектам, `negative_gradient` —
вектор **псевдоостатков** $r_i = -\partial L(y_i, a)/\partial a$ в точке $a = a_i$. Для абсолютной потери
при $y_i = a_i$ псевдоостаток равен 0. Обе функции должны давать конечный результат
при $|a_i|$ до 1000.

In [ ]:
# y: (n,) ответы, a: (n,) предсказания модели; loss: "squared_error", "absolute_error" или "log_loss"
# возвращает число: средняя потеря по объектам
def loss_value(y: np.ndarray, a: np.ndarray, loss: str) -> float:
    return ...


# y: (n,), a: (n,), loss — как выше
# возвращает (n,): псевдоостатки r_i = −∂L(y_i, a_i)/∂a_i
def negative_gradient(y: np.ndarray, a: np.ndarray, loss: str) -> np.ndarray:
    return ...

In [ ]:
# --- проверка ---
y_check = np.array([1.0, 0.0, 3.0])
a_check = np.array([0.5, 0.5, 3.0])

assert np.isclose(loss_value(y_check, a_check, "squared_error"), (0.125 + 0.125 + 0) / 3), "½(y − a)², среднее по объектам"
assert np.isclose(loss_value(y_check, a_check, "absolute_error"), (0.5 + 0.5 + 0) / 3), "|y − a|, среднее по объектам"
assert np.allclose(negative_gradient(y_check, a_check, "squared_error"), [0.5, -0.5, 0.0]), "для MSE псевдоостаток — y − a"
assert np.allclose(negative_gradient(y_check, a_check, "absolute_error"), [1.0, -1.0, 0.0]), "для MAE — знак y − a, при y = a ноль"

labels = np.array([1, 0, 1, 0])
scores = np.array([0.0, 0.0, 2.0, -1000.0])
expected_losses = np.array([np.log(2), np.log(2), np.log(1 + np.exp(-2.0)), 0.0])
assert np.isclose(loss_value(labels, scores, "log_loss"), expected_losses.mean()), "log loss: ln(1 + e^a) − y a"
assert np.allclose(negative_gradient(labels, scores, "log_loss"), labels - expit(scores)), "для log loss псевдоостаток — y − σ(a)"
assert np.isfinite(loss_value(np.array([0, 1]), np.array([1000.0, -1000.0]), "log_loss")), "при |a| = 1000 потеря должна быть конечной"

step = 1e-6
for loss in ["squared_error", "log_loss"]:
    a_point = np.array([0.3, -1.2, 2.0])
    y_point = np.array([1.0, 0.0, 1.0])
    numeric = -(loss_value(y_point, a_point + step, loss) - loss_value(y_point, a_point - step, loss)) / (2 * step) * len(a_point)
    analytic = negative_gradient(y_point, a_point, loss).sum()
    assert np.isclose(numeric, analytic, atol=1e-5), f"{loss}: псевдоостатки не совпали с численной производной"
print("проверки пройдены")

### Задача 6. Начальное приближение

Начальное приближение — лучшая константа $a_0 = \arg\min_c \sum_{i=1}^{n} L(y_i, c)$: для квадратичной
потери — среднее ответов, для абсолютной — медиана, для логистической —
$\ln\frac{\bar{y}}{1 - \bar{y}}$, где $\bar{y}$ — доля класса 1. Выводы — в конспекте, разделы 3 и 4.
Медиану четного числа значений берут как среднее двух центральных значений (`np.median`).

In [ ]:
# y: (n,) ответы; для "log_loss" — метки 0 и 1, оба класса присутствуют
# возвращает число a_0
def initial_prediction(y: np.ndarray, loss: str) -> float:
    return ...

In [ ]:
# --- проверка ---
assert np.isclose(initial_prediction(yd_fit, "squared_error"), yd_fit.mean()), "для MSE — среднее"
assert np.isclose(initial_prediction(np.array([1.0, 7.0, 2.0, 100.0]), "absolute_error"), 4.5), "для MAE — медиана"
assert np.isclose(initial_prediction(np.array([1, 1, 1, 0]), "log_loss"), np.log(3)), "для log loss — ln(доля / (1 − доля))"

constant_grid = np.linspace(yd_fit.min(), yd_fit.max(), 2001)
for loss in ["squared_error", "absolute_error"]:
    best_on_grid = constant_grid[np.argmin([loss_value(yd_fit, np.full(len(yd_fit), c), loss) for c in constant_grid])]
    assert abs(best_on_grid - initial_prediction(yd_fit, loss)) < 0.5, f"{loss}: a_0 не минимизирует потерю константой"
print("a_0 для диабета:", round(initial_prediction(yd_fit, "squared_error"), 2), round(initial_prediction(yd_fit, "absolute_error"), 2))
print("a_0 для опухолей:", round(initial_prediction(yc_fit, "log_loss"), 4))
print("проверки пройдены")

### Задача 7. Значение листа

Дерево обучают на псевдоостатках, а значение листа $R_j$ подбирают по исходной
функции потерь: $\gamma_j = \arg\min_\gamma \sum_{x_i \in R_j} L(y_i, a_i + \gamma)$. Функция получает ответы
и текущие предсказания объектов одного листа:

- квадратичная потеря — среднее остатков $y_i - a_i$;
- абсолютная — медиана остатков (`np.median`);
- логистическая — один шаг метода Ньютона
  $\gamma = \dfrac{\sum (y_i - p_i)}{\sum p_i(1 - p_i)}$, $p_i = \sigma(a_i)$; если знаменатель равен нулю, $\gamma = 0$.

In [ ]:
# y: (k,) ответы объектов листа, a: (k,) их текущие предсказания, k >= 1
# возвращает число: значение листа γ
def leaf_value(y: np.ndarray, a: np.ndarray, loss: str) -> float:
    return ...

In [ ]:
# --- проверка ---
y_leaf = np.array([3.0, 5.0, 10.0])
a_leaf = np.array([1.0, 1.0, 1.0])
assert np.isclose(leaf_value(y_leaf, a_leaf, "squared_error"), 5.0), "MSE: среднее остатков (2, 4, 9)"
assert np.isclose(leaf_value(y_leaf, a_leaf, "absolute_error"), 4.0), "MAE: медиана остатков (2, 4, 9)"

labels_leaf = np.array([1, 1, 0, 1])
scores_leaf = np.array([0.0, 0.5, -0.5, 1.0])
p_leaf = 1 / (1 + np.exp(-scores_leaf))
newton = np.sum(labels_leaf - p_leaf) / np.sum(p_leaf * (1 - p_leaf))
assert np.isclose(leaf_value(labels_leaf, scores_leaf, "log_loss"), newton), "log loss: Σ(y − p) / Σ p(1 − p)"
assert leaf_value(np.array([1, 1]), np.array([1000.0, 1000.0]), "log_loss") == 0.0, "нулевой знаменатель — значение 0"

for loss in ["squared_error", "absolute_error"]:
    shifts = np.linspace(-20, 20, 4001)
    totals = [loss_value(y_leaf, a_leaf + shift, loss) for shift in shifts]
    assert abs(shifts[int(np.argmin(totals))] - leaf_value(y_leaf, a_leaf, loss)) < 0.02, f"{loss}: γ не минимизирует потерю листа"
print("проверки пройдены")

## Блок D. Модели

### Задача 8. Регрессор

Класс `MyGradientBoostingRegressor` — градиентный бустинг для `"squared_error"`
и `"absolute_error"`. Метод `fit`:

1. `self.init_` — начальное приближение $a_0$ (задача 6); текущие предсказания на обучающей
   выборке $a_i = a_0$;
2. для $t = 1, \dots,$ `n_estimators`:
   - псевдоостатки $r_i$ в текущих предсказаниях (задача 5);
   - дерево `DecisionTreeRegressor(max_depth=self.max_depth, min_samples_leaf=self.min_samples_leaf,
     random_state=rng)`, обученное на парах $(x_i, r_i)$; генератор `rng = np.random.RandomState(self.random_state)`
     создают один раз в начале `fit` и передают всем деревьям — так деревья получают те же
     случайные числа, что в `GradientBoostingRegressor`, и при равных разбиениях выбирают
     то же разбиение;
   - для каждого листа $j$ значение $\gamma_j$ по объектам листа (задача 7); номер листа
     объекта возвращает метод дерева `apply`;
   - $a_i \leftarrow a_i + \eta\,\gamma_{j(i)}$, где $j(i)$ — лист объекта $i$, $\eta$ — `learning_rate`;
   - дерево добавляют в список `self.estimators_`, словарь {номер листа: $\gamma_j$} —
     в список `self.leaf_values_`, среднюю потерю на обучающей выборке после
     этого дерева — в `self.train_loss_[t - 1]`.

Сами деревья не меняют: их `predict` по-прежнему возвращает среднее псевдоостатков
листа, а значения $\gamma_j$ хранятся отдельно. Метод `predict` возвращает
$a_0 + \eta\sum_t \gamma_{t, j_t(x)}$, `staged_predict` — предсказания после каждого дерева.

In [ ]:
class MyGradientBoostingRegressor:
    def __init__(self, n_estimators: int = 100, learning_rate: float = 0.1, max_depth: int = 3,
                 min_samples_leaf: int = 1, loss: str = "squared_error", random_state: int | None = None):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.loss = loss
        self.random_state = random_state

    # X: (n, p), y: (n,); задает атрибуты и возвращает self:
    #   init_: float — a_0
    #   estimators_: list[DecisionTreeRegressor] длины n_estimators — деревья в порядке обучения
    #   leaf_values_: list[dict[int, float]] длины n_estimators — {номер листа из apply: γ}
    #   train_loss_: np.ndarray (n_estimators,) — средняя потеря на обучающей выборке после каждого дерева
    def fit(self, X: np.ndarray, y: np.ndarray) -> "MyGradientBoostingRegressor":
        ...
        return self

    # X: (m, p); возвращает (n_estimators, m): строка t — предсказания после первых t + 1 деревьев
    def staged_predict(self, X: np.ndarray) -> np.ndarray:
        return ...

    # X: (m, p); возвращает (m,): предсказания после всех деревьев
    def predict(self, X: np.ndarray) -> np.ndarray:
        return ...

In [ ]:
# --- проверка ---
mine = MyGradientBoostingRegressor(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=SEED)
mine.fit(Xd_fit, yd_fit)

reference = GradientBoostingRegressor(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=SEED)
reference.fit(Xd_fit, yd_fit)

assert len(mine.estimators_) == 50 and len(mine.leaf_values_) == 50, "по дереву и словарю значений на каждую итерацию"
assert np.isclose(mine.init_, yd_fit.mean()), "init_ — среднее ответов"
assert np.allclose(mine.predict(Xd_fit), reference.predict(Xd_fit)), "MSE: предсказания на обучающей выборке не совпали с sklearn"

stages = mine.staged_predict(Xd_val)
assert stages.shape == (50, len(Xd_val)), "staged_predict возвращает (n_estimators, m)"
assert np.allclose(stages[-1], mine.predict(Xd_val)), "последняя строка staged_predict — predict"
assert np.all(np.diff(mine.train_loss_) <= 1e-9), "для MSE потеря на обучении не растет"

robust = MyGradientBoostingRegressor(n_estimators=30, loss="absolute_error", random_state=SEED)
robust.fit(Xd_fit, yd_fit)
assert np.isclose(robust.init_, np.median(yd_fit)), "для MAE init_ — медиана"

first_tree = robust.estimators_[0]
first_leaves = first_tree.apply(Xd_fit)
some_leaf = int(first_leaves[0])
residuals_in_leaf = (yd_fit - robust.init_)[first_leaves == some_leaf]
signs_in_leaf = np.sign(yd_fit - robust.init_)[first_leaves == some_leaf]
assert np.isclose(robust.leaf_values_[0][some_leaf], np.median(residuals_in_leaf)), "для MAE значение листа — медиана остатков"
assert np.isclose(first_tree.predict(Xd_fit[:1])[0], signs_in_leaf.mean()), "дерево обучено на знаках остатков и не изменено"

val_mse = np.mean((mine.predict(Xd_val) - yd_val) ** 2)
print(f"MSE на валидации: свой бустинг {val_mse:.0f}, sklearn {np.mean((reference.predict(Xd_val) - yd_val) ** 2):.0f}")
print("проверки пройдены")

### Задача 9. Бинарный классификатор

Класс `MyGradientBoostingClassifier` — бустинг с логистической потерей для меток
0 и 1. Алгоритм `fit` и атрибуты те же, что в задаче 8, с `loss="log_loss"`;
можно унаследовать класс задачи 8. Методы:

- `decision_function` — оценка $a(x)$ после всех деревьев;
- `staged_decision_function` — оценки после каждого дерева;
- `predict_proba` — вероятности классов 0 и 1: $(1 - \sigma(a),\ \sigma(a))$;
- `predict` — класс с большей вероятностью, при $\sigma(a) = 0.5$ — класс 0.

In [ ]:
class MyGradientBoostingClassifier:
    def __init__(self, n_estimators: int = 100, learning_rate: float = 0.1, max_depth: int = 3,
                 min_samples_leaf: int = 1, random_state: int | None = None):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.random_state = random_state

    # X: (n, p), y: (n,) метки 0 и 1; задает init_, estimators_, leaf_values_, train_loss_ как в задаче 8
    def fit(self, X: np.ndarray, y: np.ndarray) -> "MyGradientBoostingClassifier":
        ...
        return self

    # X: (m, p); возвращает (n_estimators, m): оценки a(x) после первых t + 1 деревьев
    def staged_decision_function(self, X: np.ndarray) -> np.ndarray:
        return ...

    # X: (m, p); возвращает (m,): оценки a(x) после всех деревьев
    def decision_function(self, X: np.ndarray) -> np.ndarray:
        return ...

    # X: (m, p); возвращает (m, 2): вероятности классов 0 и 1
    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        return ...

    # X: (m, p); возвращает (m,): метки 0 и 1
    def predict(self, X: np.ndarray) -> np.ndarray:
        return ...

In [ ]:
# --- проверка ---
mine_clf = MyGradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=SEED)
mine_clf.fit(Xc_fit, yc_fit)

reference_clf = GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=SEED)
reference_clf.fit(Xc_fit, yc_fit)

share = yc_fit.mean()
assert np.isclose(mine_clf.init_, np.log(share / (1 - share))), "init_ — логарифм отношения шансов"
assert np.allclose(mine_clf.decision_function(Xc_fit), reference_clf.decision_function(Xc_fit)), "оценки на обучающей выборке не совпали с sklearn"

probabilities = mine_clf.predict_proba(Xc_val)
assert probabilities.shape == (len(Xc_val), 2) and np.allclose(probabilities.sum(axis=1), 1), "predict_proba — (m, 2), строки в сумме 1"
assert np.array_equal(mine_clf.predict(Xc_val), (probabilities[:, 1] > 0.5).astype(int)), "predict — класс с большей вероятностью"
assert mine_clf.staged_decision_function(Xc_val).shape == (50, len(Xc_val)), "staged_decision_function — (n_estimators, m)"
assert mine_clf.train_loss_[-1] < mine_clf.train_loss_[0], "потеря на обучении должна уменьшаться"

val_accuracy = np.mean(mine_clf.predict(Xc_val) == yc_val)
print(f"accuracy на валидации: {val_accuracy:.3f}, log loss на обучении после 50 деревьев: {mine_clf.train_loss_[-1]:.4f}")
print("проверки пройдены")

## Блок E. Эксперименты

### Задача 10. Темп обучения и число деревьев

Для каждого темпа из `RATES` обучите `MyGradientBoostingRegressor(n_estimators=N_TREES,
learning_rate=темп, max_depth=3, random_state=SEED)` на `Xd_fit` и по `staged_predict`
найдите число деревьев с наименьшей MSE (mean squared error, среднеквадратичная
ошибка) $\frac{1}{m}\sum_i (y_i - a_i)^2$ на валидационной части `Xd_val` (при равенстве — меньшее);
это среднее квадратов без множителя $\frac{1}{2}$, а не `loss_value`. В таблицу `rate_table` запишите это число,
MSE на валидации и MSE на тестовом сете при этом числе деревьев. Индекс —
темпы из `RATES` с именем `"темп"`, столбцы — `RATE_COLUMNS`. Нарисуйте кривые
MSE на валидации от числа деревьев для всех темпов на одном графике.

In [ ]:
RATES = [1.0, 0.3, 0.1, 0.03]
N_TREES = 500
RATE_COLUMNS = ["лучшее число деревьев", "MSE на валидации", "MSE на тесте"]

In [ ]:
rate_table: pd.DataFrame = ...     # (len(RATES), 3): индекс "темп", столбцы RATE_COLUMNS

In [ ]:
# --- проверка ---
assert list(rate_table.index) == RATES and rate_table.index.name == "темп", "индекс — RATES с именем «темп»"
assert list(rate_table.columns) == RATE_COLUMNS, "столбцы — RATE_COLUMNS в том же порядке"
assert rate_table["лучшее число деревьев"].between(1, N_TREES).all(), "число деревьев от 1 до N_TREES"
assert rate_table.loc[1.0, "лучшее число деревьев"] < rate_table.loc[0.03, "лучшее число деревьев"], "при меньшем темпе нужно больше деревьев"
print(rate_table.round(1))
print("проверки пройдены")

**Вопрос.** Почему при $\eta = 1$ лучшее число деревьев мало, а при $\eta = 0.03$ велико?
Совпадает ли лучший темп по валидации с лучшим по тестовому сету, и почему
выбирать параметры по тестовому сету нельзя?

*Ответ пишите здесь.*

### Задача 11. Ранняя остановка

Ранняя остановка с терпением $k$: деревья добавляют по одному, после $t$-го дерева
известна потеря на валидации $L_t$. Пусть $t^*$ — номер дерева с наименьшей потерей
среди $L_1, \dots, L_t$ (при равенстве — меньший номер). Обучение прекращают на первом $t$,
при котором $t - t^* \ge k$: за $k$ деревьев потеря ни разу не стала меньше лучшей.
Функция `early_stopping_trees` возвращает $t^*$ на момент остановки — число деревьев,
которое оставляют в модели; если остановки не было, — $t^*$ по всем деревьям.

Затем обучите `MyGradientBoostingClassifier(n_estimators=500, learning_rate=0.1, max_depth=3,
random_state=SEED)` на `Xc_fit`, вычислите log loss на `Xc_val` после каждого дерева
(`staged_decision_function` и `loss_value`) и найдите `stopped_trees` с терпением
`PATIENCE`. Запишите в словарь `early_stopping_result` log loss на тестовом сете
у модели из `stopped_trees` деревьев и из всех 500. Нарисуйте log loss на валидации
и на тестовом сете от числа деревьев и отметьте `stopped_trees`.

In [ ]:
PATIENCE = 20

In [ ]:
# validation_losses: (T,) потери на валидации после 1, ..., T деревьев; patience >= 1
# возвращает int от 1 до T: число деревьев, которое оставляет ранняя остановка
def early_stopping_trees(validation_losses: np.ndarray, patience: int) -> int:
    return ...


stopped_trees: int = ...
early_stopping_result: dict[str, float] = ...   # {"log loss на тесте при остановке": число, "log loss на тесте при 500 деревьях": число}

In [ ]:
# --- проверка ---
assert early_stopping_trees(np.array([5.0, 4.0, 3.0, 2.0]), 2) == 4, "потеря все время падает — все деревья"
assert early_stopping_trees(np.array([3.0, 1.0, 2.0, 2.0, 0.5]), 2) == 2, "после второго дерева два дерева без улучшения — остановка"
assert early_stopping_trees(np.array([3.0, 1.0, 2.0, 0.5, 2.0]), 2) == 4, "улучшение на четвертом дереве сбрасывает отсчет"
assert early_stopping_trees(np.array([2.0, 2.0, 2.0]), 5) == 1, "равенство — не улучшение, при равных — меньший номер"
assert 1 <= stopped_trees <= 500, "stopped_trees — от 1 до 500"
assert set(early_stopping_result) == {"log loss на тесте при остановке", "log loss на тесте при 500 деревьях"}, "ключи early_stopping_result"
print("остановка на", stopped_trees, "деревьях;", {name: round(value, 4) for name, value in early_stopping_result.items()})
print("проверки пройдены")

### Задача 12. Сравнение в одном сетапе

Обучите на `Xc_train` (обучение и валидация вместе) модели из `MODEL_NAMES`
с параметрами из функции `make_models` и запишите в таблицу `models_table`
log loss, ROC AUC (receiver operating characteristic, area under the curve — площадь
под ROC-кривой) и accuracy на тестовом сете `Xc_test` и время обучения
в секундах (`time.perf_counter`). Индекс — `MODEL_NAMES`, столбцы — `MODEL_COLUMNS`.
Метрики считают функциями `log_loss`, `roc_auc_score` по вероятности класса 1
и `accuracy_score` по `predict`.

In [ ]:
MODEL_COLUMNS = ["log loss", "ROC AUC", "accuracy", "время обучения, с"]


def make_models() -> dict:
    """Модели сравнения: 200 деревьев глубины 3 и темп 0.1 у всех вариантов бустинга."""
    models = {
        "дерево, глубина 3": DecisionTreeClassifier(max_depth=3, random_state=SEED),
        "случайный лес": RandomForestClassifier(n_estimators=300, random_state=SEED),
        "свой бустинг": MyGradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=3, random_state=SEED),
        "GradientBoostingClassifier": GradientBoostingClassifier(
            n_estimators=200, learning_rate=0.1, max_depth=3, random_state=SEED),
        "HistGradientBoostingClassifier": HistGradientBoostingClassifier(
            max_iter=200, learning_rate=0.1, max_depth=3, early_stopping=False, random_state=SEED),
        "LightGBM": LGBMClassifier(
            n_estimators=200, learning_rate=0.1, max_depth=3, num_leaves=8, random_state=SEED, verbose=-1),
        "CatBoost": CatBoostClassifier(
            iterations=200, learning_rate=0.1, depth=3, random_seed=SEED, verbose=0, allow_writing_files=False),
    }
    return models


MODEL_NAMES = list(make_models())

In [ ]:
models_table: pd.DataFrame = ...     # (len(MODEL_NAMES), 4): индекс MODEL_NAMES, столбцы MODEL_COLUMNS

In [ ]:
# --- проверка ---
assert list(models_table.index) == MODEL_NAMES and list(models_table.columns) == MODEL_COLUMNS, "индекс — MODEL_NAMES, столбцы — MODEL_COLUMNS"
assert models_table["ROC AUC"].between(0.5, 1).all() and models_table["accuracy"].between(0.5, 1).all(), "метрики качества от 0.5 до 1"
assert (models_table["время обучения, с"] > 0).all(), "время обучения положительное"

reference_tree = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(Xc_train, yc_train)
expected_loss = log_loss(yc_test, reference_tree.predict_proba(Xc_test)[:, 1])
assert np.isclose(models_table.loc["дерево, глубина 3", "log loss"], expected_loss), "log loss дерева не совпал с пересчетом"

models_table.style.format(precision=4).background_gradient(cmap="Greens", subset=["ROC AUC", "accuracy"]) \
    .background_gradient(cmap="Greens_r", subset=["log loss", "время обучения, с"])

**Вопрос.** На обучающей выборке оценки своего бустинга и `GradientBoostingClassifier`
совпадают (задача 9), а метрики на тестовом сете немного различаются. Найдите
первое дерево, в котором модели расходятся (атрибуты `tree_.feature` и `tree_.threshold`
у деревьев `estimators_`), и объясните, почему расхождение не меняет предсказаний
на обучающей выборке. Почему LightGBM и CatBoost с тем же числом и глубиной деревьев
дают другой log loss? Какая модель лучше по log loss и какая по accuracy и почему эти
метрики выбирают разные модели?

*Ответ пишите здесь.*

## Бонусы

### Задача 13, бонус. Многоклассовый бустинг

Для $K$ классов модель хранит $K$ оценок $a_1(x), \dots, a_K(x)$, вероятности — softmax:
$p_k = e^{a_k} / \sum_{m} e^{a_m}$. Потеря — $-\ln p_{y}$; псевдоостаток для оценки $k$ равен $[y_i = k] - p_{ik}$.
На каждой итерации строят $K$ деревьев, дерево $k$ — на псевдоостатках класса $k$;
все $K$ деревьев итерации используют вероятности, посчитанные до этой итерации.
Значение листа дерева $k$ (Friedman, 2001):

$$\gamma = \frac{K - 1}{K}\cdot\frac{\sum_{x_i \in R} r_{ik}}{\sum_{x_i \in R} p_{ik}(1 - p_{ik})},$$

при нулевом знаменателе $\gamma = 0$. Начальное приближение $a_{0k} = \ln \bar{y}_k$, где $\bar{y}_k$ — доля класса $k$.
Метки — целые от 0 до $K - 1$, все классы присутствуют в обучающей выборке.
Деревья — как в задаче 8, с одним генератором `rng` на все $K$ деревьев всех итераций
в порядке «итерация, затем класс».

In [ ]:
class MyGradientBoostingMulticlass:
    def __init__(self, n_estimators: int = 100, learning_rate: float = 0.1, max_depth: int = 3,
                 min_samples_leaf: int = 1, random_state: int | None = None):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.random_state = random_state

    # X: (n, p), y: (n,) метки 0..K-1; задает атрибуты и возвращает self:
    #   init_: np.ndarray (K,) — a_0k
    #   estimators_: list длины n_estimators, элемент — list из K деревьев итерации
    #   leaf_values_: list длины n_estimators, элемент — list из K словарей {номер листа: γ}
    def fit(self, X: np.ndarray, y: np.ndarray) -> "MyGradientBoostingMulticlass":
        ...
        return self

    # X: (m, p); возвращает (m, K): оценки a_k(x)
    def decision_function(self, X: np.ndarray) -> np.ndarray:
        return ...

    # X: (m, p); возвращает (m, K): вероятности классов
    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        return ...

    # X: (m, p); возвращает (m,): метки
    def predict(self, X: np.ndarray) -> np.ndarray:
        return ...

In [ ]:
# --- проверка ---
mine_multi = MyGradientBoostingMulticlass(n_estimators=30, learning_rate=0.1, max_depth=2, random_state=SEED)
mine_multi.fit(Xw_train, yw_train)

reference_multi = GradientBoostingClassifier(n_estimators=30, learning_rate=0.1, max_depth=2, random_state=SEED)
reference_multi.fit(Xw_train, yw_train)

assert len(mine_multi.estimators_) == 30 and all(len(trees) == 3 for trees in mine_multi.estimators_), "по три дерева на итерацию"
assert np.allclose(mine_multi.predict_proba(Xw_train), reference_multi.predict_proba(Xw_train)), "вероятности на обучении не совпали с sklearn"
assert mine_multi.predict_proba(Xw_test).shape == (len(Xw_test), 3), "predict_proba — (m, K)"
print("accuracy на тесте:", np.mean(mine_multi.predict(Xw_test) == yw_test).round(3))
print("проверки пройдены")

### Задача 14, бонус. Разбиение по второй производной

XGBoost и LightGBM выбирают разбиения и значения листьев по первым производным
$g_i = \partial L/\partial a$ и вторым $h_i = \partial^2 L/\partial a^2$ потери в текущих предсказаниях
с регуляризацией $\lambda$. Для листа с суммами $G = \sum g_i$ и $H = \sum h_i$ значение
$w = -\frac{G}{H + \lambda}$; прирост разбиения на потомков $l$ и $r$ со штрафом $\mu$ за лист:

$$\text{Gain} = \frac{1}{2}\Big(\frac{G_l^2}{H_l + \lambda} + \frac{G_r^2}{H_r + \lambda} - \frac{(G_l + G_r)^2}{H_l + H_r + \lambda}\Big) - \mu.$$

Вывод — в конспекте. Функция `best_second_order_threshold` перебирает пороги
по одному признаку — середины между соседними различными значениями — и возвращает
порог с наибольшим приростом среди тех, где в обоих потомках сумма $h_i$ не меньше
`min_child_weight` (при равенстве приростов — меньший порог), и сам прирост;
если допустимых порогов нет — `None`. Объекты с $x_i \le$ порога идут влево.

In [ ]:
# g, h: (k,) первые и вторые производные потери объектов листа; reg_lambda >= 0
# возвращает число w
def second_order_leaf_weight(g: np.ndarray, h: np.ndarray, reg_lambda: float) -> float:
    return ...


# g, h: (n,), left_mask: (n,) bool — True у объектов левого потомка; gamma — штраф μ
# возвращает число Gain
def second_order_gain(g: np.ndarray, h: np.ndarray, left_mask: np.ndarray, reg_lambda: float, gamma: float) -> float:
    return ...


# x: (n,) значения признака, g, h: (n,); возвращает (порог, прирост) или None
def best_second_order_threshold(x: np.ndarray, g: np.ndarray, h: np.ndarray, reg_lambda: float,
                                gamma: float, min_child_weight: float) -> tuple[float, float] | None:
    return ...

In [ ]:
# --- проверка ---
g_toy = np.array([-1.0, -1.0, 1.0, 1.0])
h_toy = np.array([1.0, 1.0, 1.0, 1.0])
assert np.isclose(second_order_leaf_weight(g_toy[:2], h_toy[:2], 0.0), 1.0), "w = −G / (H + λ)"
assert np.isclose(second_order_leaf_weight(g_toy[:2], h_toy[:2], 2.0), 0.5), "λ уменьшает значение листа"
split_toy = np.array([True, True, False, False])
assert np.isclose(second_order_gain(g_toy, h_toy, split_toy, 0.0, 0.0), 0.5 * (4 / 2 + 4 / 2 - 0)), "прирост по формуле"
assert np.isclose(second_order_gain(g_toy, h_toy, split_toy, 0.0, 1.5), 0.5), "штраф μ вычитается из прироста"

found = best_second_order_threshold(np.array([3.0, 1.0, 4.0, 2.0]), np.array([1.0, -1.0, 1.0, -1.0]), h_toy, 0.0, 0.0, 0.0)
assert found is not None and np.isclose(found[0], 2.5) and np.isclose(found[1], 2.0), "порог 2.5 разделяет g = −1 и g = 1"
assert best_second_order_threshold(np.ones(4), g_toy, h_toy, 1.0, 0.0, 0.0) is None, "постоянный признак — None"

# пень LightGBM с регуляризацией λ: значения его листьев — −G / (H + λ) при a = 0
stump_lambda = 3.0
lgbm_stump = LGBMClassifier(n_estimators=1, learning_rate=1.0, num_leaves=2, reg_lambda=stump_lambda, min_child_samples=1,
                            min_child_weight=0, boost_from_average=False, verbose=-1)
lgbm_stump.fit(Xc_fit, yc_fit)
leaf_index = lgbm_stump.predict(Xc_fit, pred_leaf=True).ravel()
raw_scores = lgbm_stump.predict(Xc_fit, raw_score=True)
gradients = 0.5 - yc_fit                                 # g = p − y при a = 0, p = 0.5
hessians = np.full(len(yc_fit), 0.25)                    # h = p(1 − p)
for leaf in np.unique(leaf_index):
    in_leaf = leaf_index == leaf
    mine_weight = second_order_leaf_weight(gradients[in_leaf], hessians[in_leaf], stump_lambda)
    assert np.isclose(mine_weight, raw_scores[in_leaf][0]), "значение листа не совпало с LightGBM"
print("проверки пройдены")

### Задача 15, бонус. Квантильная регрессия

Квантиль уровня $\alpha$ условного распределения $y$ минимизирует **функцию
потерь пинбола** (pinball loss):

$$L_\alpha(y, a) = \begin{cases} \alpha\,(y - a), & y \ge a, \\ (1 - \alpha)(a - y), & y < a. \end{cases}$$

Псевдоостаток $r = \alpha$ при $y \ge a$ и $r = \alpha - 1$ при $y < a$; начальное приближение
и значение листа — квантиль уровня $\alpha$ ответов и остатков листа (`np.quantile`
с параметрами по умолчанию). Класс `MyQuantileBoostingRegressor` устроен как в задаче 8
с этой потерей; `train_loss_` не нужен. Затем на диабете обучите на `Xd_train` две
модели, $\alpha = 0.1$ и $\alpha = 0.9$, по 200 деревьев глубины 3 с темпом 0.1 и `random_state=SEED`,
и вычислите `interval_coverage` — долю объектов тестового сета, у которых ответ лежит
между двумя предсказаниями, включая границы, — и `interval_width` — среднюю разность предсказаний
верхней и нижней модели.

In [ ]:
# y, a: (n,); возвращает число: средняя потеря пинбола
def pinball_loss(y: np.ndarray, a: np.ndarray, alpha: float) -> float:
    return ...


class MyQuantileBoostingRegressor:
    def __init__(self, alpha: float = 0.5, n_estimators: int = 100, learning_rate: float = 0.1, max_depth: int = 3,
                 min_samples_leaf: int = 1, random_state: int | None = None):
        self.alpha = alpha
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.random_state = random_state

    # X: (n, p), y: (n,); задает init_, estimators_, leaf_values_ как в задаче 8, возвращает self
    def fit(self, X: np.ndarray, y: np.ndarray) -> "MyQuantileBoostingRegressor":
        ...
        return self

    # X: (m, p); возвращает (m,)
    def predict(self, X: np.ndarray) -> np.ndarray:
        return ...


interval_coverage: float = ...     # доля объектов тестового сета внутри интервала
interval_width: float = ...        # средняя ширина интервала

In [ ]:
# --- проверка ---
assert np.isclose(pinball_loss(np.array([1.0, 3.0]), np.array([2.0, 2.0]), 0.9), (0.1 * 1 + 0.9 * 1) / 2), "потеря пинбола по формуле"

median_model = MyQuantileBoostingRegressor(alpha=0.5, n_estimators=20, random_state=SEED)
median_model.fit(Xd_fit, yd_fit)
assert np.isclose(median_model.init_, np.quantile(yd_fit, 0.5)), "init_ — квантиль ответов"
assert pinball_loss(yd_fit, median_model.predict(Xd_fit), 0.5) < pinball_loss(yd_fit, np.full(len(yd_fit), median_model.init_), 0.5), \
    "бустинг должен уменьшать потерю на обучении"

assert 0 <= interval_coverage <= 1 and interval_width > 0, "доля от 0 до 1, ширина положительная"
print(f"интервал от квантиля 0.1 до квантиля 0.9: покрытие {interval_coverage:.3f}, средняя ширина {interval_width:.1f}")
print("проверки пройдены")